# Topic Modeling with BERTopic (HDBSCAN + UMAP)

Loads pre-computed embeddings for the **iGEM teams** and **SynBio papers** datasets,
fits a BERTopic model (UMAP → HDBSCAN) on each, displays summary info, and saves
the fitted models using BERTopic's built-in serialisation.

In [ ]:
import os
import numpy as np
import pandas as pd
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
ASSETS_DIR     = os.path.join('..', 'assets')
EMBEDDINGS_DIR = os.path.join(ASSETS_DIR, 'embeddings')
MODELS_DIR     = os.path.join(ASSETS_DIR, 'topic_models')
os.makedirs(MODELS_DIR, exist_ok=True)

# HDBSCAN minimum cluster size (tune per dataset)
MIN_CLUSTER_SIZE_TEAMS  = 10
MIN_CLUSTER_SIZE_PAPERS = 30

## 1. Load embeddings and corpora

In [ ]:
teams_embeddings  = np.load(os.path.join(EMBEDDINGS_DIR, 'teams_embeddings.npy'))
papers_embeddings = np.load(os.path.join(EMBEDDINGS_DIR, 'papers_embeddings.npy'))

teams_corpus  = pd.read_csv(os.path.join(EMBEDDINGS_DIR, 'teams_corpus.txt'), sep='\t')
papers_corpus = pd.read_csv(os.path.join(EMBEDDINGS_DIR, 'papers_corpus.txt'), sep='\t')

teams_docs  = teams_corpus['text'].tolist()
papers_docs = papers_corpus['text'].tolist()

print(f'Teams:  {teams_embeddings.shape[0]:,} docs, {teams_embeddings.shape[1]} dims')
print(f'Papers: {papers_embeddings.shape[0]:,} docs, {papers_embeddings.shape[1]} dims')

assert len(teams_docs) == teams_embeddings.shape[0]
assert len(papers_docs) == papers_embeddings.shape[0]

## 2. Helper: build and fit a topic model

In [ ]:
def fit_topic_model(
    docs: list[str],
    embeddings: np.ndarray,
    min_cluster_size: int = 15,
    umap_n_neighbors: int = 15,
    umap_n_components: int = 5,
) -> tuple[BERTopic, list[int], np.ndarray]:
    """Fit a BERTopic model using UMAP + HDBSCAN."""
    umap_model = UMAP(
        n_neighbors=umap_n_neighbors,
        n_components=umap_n_components,
        min_dist=0.0,
        metric='cosine',
        random_state=SEED,
    )
    hdbscan_model = HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=1,
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True,
    )
    topic_model = BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        min_topic_size=min_cluster_size,
        n_gram_range=(1, 3),
        language='english',
        calculate_probabilities=True,
        verbose=True,
    )
    topics, probs = topic_model.fit_transform(docs, embeddings)
    return topic_model, topics, probs

## 3. Fit: iGEM Teams

In [ ]:
teams_model, teams_topics, teams_probs = fit_topic_model(
    teams_docs, teams_embeddings, min_cluster_size=MIN_CLUSTER_SIZE_TEAMS
)

In [ ]:
teams_info = teams_model.get_topic_info()
print(f'Teams — topics found: {teams_info.Topic.max() + 1}  |  outliers: {(np.array(teams_topics) == -1).sum():,}')
teams_info.head(20)

## 4. Fit: SynBio Papers

In [ ]:
papers_model, papers_topics, papers_probs = fit_topic_model(
    papers_docs, papers_embeddings, min_cluster_size=MIN_CLUSTER_SIZE_PAPERS
)

In [ ]:
papers_info = papers_model.get_topic_info()
print(f'Papers — topics found: {papers_info.Topic.max() + 1}  |  outliers: {(np.array(papers_topics) == -1).sum():,}')
papers_info.head(20)

## 5. Save topic models and summaries

In [ ]:
# Save models using BERTopic's built-in method
teams_model.save(os.path.join(MODELS_DIR, 'teams_topic_model'))
papers_model.save(os.path.join(MODELS_DIR, 'papers_topic_model'))

# Save topic info summaries as TSV
teams_info.to_csv(os.path.join(MODELS_DIR, 'teams_topic_info.txt'), sep='\t', index=False)
papers_info.to_csv(os.path.join(MODELS_DIR, 'papers_topic_info.txt'), sep='\t', index=False)

# Save document-level topic assignments
teams_corpus['topic'] = teams_topics
teams_corpus[['UT', 'topic']].to_csv(
    os.path.join(MODELS_DIR, 'teams_doc_topics.txt'), sep='\t', index=False
)
papers_corpus['topic'] = papers_topics
papers_corpus[['id', 'topic']].to_csv(
    os.path.join(MODELS_DIR, 'papers_doc_topics.txt'), sep='\t', index=False
)

print(f'All outputs saved to {MODELS_DIR}')
for f in sorted(os.listdir(MODELS_DIR)):
    print(f'  {f}')